# Lab 0 - CNN Experiments on CIFAR-10

| Experiment | Optimizer | Activation | Test Accuracy |
|---|---|---|---|
| Baseline | SGD (lr=0.0001) | LeakyReLU | - |
| Experiment 1 | Adam | LeakyReLU | - |
| Experiment 2 | Adam | Tanh | - |

## 1. Imports & Setup

In [ ]:
%%time

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import numpy as np

# Limit GPU memory to 25% since we are sharing with 4 people
torch.cuda.set_per_process_memory_fraction(0.25, device=0)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 2. Load & Prepare CIFAR-10

In [ ]:
%%time

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Download and load training set
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True, num_workers=2
)

# Download and load test set
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False, num_workers=2
)

# CIFAR-10 class labels
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print('Training samples:', len(trainset))
print('Test samples:', len(testset))

## 3. Define CNN Model & Helper Functions (Reusable)

In [ ]:
%%time

class CNN(nn.Module):
    def __init__(self, activation_fn=nn.LeakyReLU()):
        super(CNN, self).__init__()
        self.activation_fn = activation_fn

        # Convolutional layers
        self.conv_block = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),                          # 32x16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),                          # 64x8x8

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2)                           # 128x4x4
        )

        # Fully connected layers
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            self.activation_fn,
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x


def train_model(model, optimizer, num_epochs, run_name):
    """Reusable training function for all experiments."""
    criterion = nn.CrossEntropyLoss()
    writer = SummaryWriter(f'runs/{run_name}')

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(trainloader)
        epoch_acc = 100. * correct / total

        # Log to Tensorboard
        writer.add_scalar('Loss/train', epoch_loss, epoch)
        writer.add_scalar('Accuracy/train', epoch_acc, epoch)

        print(f'Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%')

    writer.close()
    print('Training complete!')
    return model


def evaluate_model(model, label):
    """Reusable evaluation function for all experiments."""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    accuracy = 100. * correct / total
    print(f'Test Accuracy ({label}): {accuracy:.2f}%')
    return accuracy


# Store results for final summary
results = {}
NUM_EPOCHS = 10

print('Model and helper functions defined!')

---
## Baseline: SGD + LeakyReLU

In [ ]:
%%time

# Instantiate model with LeakyReLU
model_baseline = CNN(activation_fn=nn.LeakyReLU()).to(device)

# SGD optimizer with lr=0.0001
optimizer_baseline = optim.SGD(model_baseline.parameters(), lr=0.0001)

print('Optimizer: SGD | LR: 0.0001 | Activation: LeakyReLU')
print('-' * 55)

model_baseline = train_model(model_baseline, optimizer_baseline, NUM_EPOCHS, 'baseline_sgd_leakyrelu')

In [ ]:
%%time

results['Baseline (SGD + LeakyReLU)'] = evaluate_model(model_baseline, 'Baseline - SGD + LeakyReLU')

---
## Experiment 1: Adam + LeakyReLU

In [ ]:
%%time

# Instantiate a fresh model with LeakyReLU
model_exp1 = CNN(activation_fn=nn.LeakyReLU()).to(device)

# Adam optimizer (default lr=0.001)
optimizer_exp1 = optim.Adam(model_exp1.parameters())

print('Optimizer: Adam | Activation: LeakyReLU')
print('-' * 55)

model_exp1 = train_model(model_exp1, optimizer_exp1, NUM_EPOCHS, 'exp1_adam_leakyrelu')

In [ ]:
%%time

results['Experiment 1 (Adam + LeakyReLU)'] = evaluate_model(model_exp1, 'Experiment 1 - Adam + LeakyReLU')

---
## Experiment 2: Adam + Tanh

In [ ]:
%%time

# Instantiate a fresh model with Tanh
model_exp2 = CNN(activation_fn=nn.Tanh()).to(device)

# Adam optimizer (default lr=0.001)
optimizer_exp2 = optim.Adam(model_exp2.parameters())

print('Optimizer: Adam | Activation: Tanh')
print('-' * 55)

model_exp2 = train_model(model_exp2, optimizer_exp2, NUM_EPOCHS, 'exp2_adam_tanh')

In [ ]:
%%time

results['Experiment 2 (Adam + Tanh)'] = evaluate_model(model_exp2, 'Experiment 2 - Adam + Tanh')

---
## Results Summary

In [ ]:
print('=' * 55)
print(f'{"Experiment":<35} {"Test Accuracy":>15}')
print('=' * 55)
for name, acc in results.items():
    print(f'{name:<35} {acc:>14.2f}%')
print('=' * 55)

## Tensorboard

In [ ]:
# To launch Tensorboard, open a terminal in Jupyter and run:
# tensorboard --logdir runs --host 0.0.0.0 --port 6006
# Then open in browser: https://adl-10.icedc.se/proxy/6006
print('To launch Tensorboard, open a terminal and run:')
print('  tensorboard --logdir runs --host 0.0.0.0 --port 6006')
print('Then open in browser: https://adl-10.icedc.se/proxy/6006')